# 02 — Model Fitting

This notebook:
1. Fits the inhomogeneous Poisson scoring-intensity model.
2. Fits the Hawkes process to significant odds-change events.
3. Runs the Kalman filter on a sample game.
4. Runs Monte Carlo simulation from mid-game state.
5. Visualizes the filtered WP path and Hawkes intensity.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

with open('../config/default.yaml') as f:
    cfg = yaml.safe_load(f)

print('Ready.')

## 2.1 Load Data and Build Features

In [ ]:
from src.data_ingestion.play_by_play import load_pbp
from src.data_ingestion.odds_simulator import SyntheticOddsGenerator
from src.feature_engineering.game_state_features import build_game_state_features
from src.feature_engineering.odds_path_features import build_odds_path_features

pbp = load_pbp(seasons=[2022], cache_dir='../data/processed')
pbp_feat = build_game_state_features(pbp)

# Generate synthetic odds for all games
gen = SyntheticOddsGenerator(seed=42)
odds_all = gen.generate_multiple_games(pbp)
odds_feat_all = build_odds_path_features(odds_all)

print(f'PBP: {len(pbp_feat):,} rows, {pbp_feat["game_id"].nunique()} games')
print(f'Odds: {len(odds_all):,} rows')

## 2.2 Fit the Poisson Scoring Intensity Model

In [ ]:
from src.models.poisson_scoring import ScoringIntensityModel

poisson_cfg = cfg['poisson_scoring']
scoring_model = ScoringIntensityModel(
    avg_seconds_per_play=poisson_cfg['avg_seconds_per_play'],
    logistic_C=poisson_cfg['logistic_C'],
    logistic_max_iter=poisson_cfg['logistic_max_iter'],
).fit(pbp_feat)

# In-sample scoring probability distribution
probs = scoring_model.predict_proba(pbp_feat)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(probs, bins=50, color='#1f77b4', edgecolor='white')
ax.set_xlabel('Predicted P(Scoring Event | Game State)')
ax.set_ylabel('Count')
ax.set_title('Poisson Scoring Model — Predicted Probabilities')
plt.tight_layout()
plt.show()

print(f'Mean predicted scoring prob: {probs.mean():.4f}')
print(f'Actual scoring rate: {(pbp_feat["touchdown"].fillna(0) > 0).mean():.4f}')

## 2.3 Fit the Hawkes Process

In [ ]:
from src.models.hawkes_process import HawkesOddsModel

hcfg = cfg['hawkes']
hawkes_model = HawkesOddsModel(
    threshold=hcfg['significant_move_threshold'],
    mu_init=hcfg['mu_init'],
    alpha_init=hcfg['alpha_init'],
    beta_init=hcfg['beta_init'],
)

# Use a single game for illustration
sample_gid = pbp_feat['game_id'].unique()[0]
game_odds = odds_feat_all[odds_feat_all['game_id'] == sample_gid].copy()

# Significant odds moves
event_mask = game_odds['odds_return'].abs() > hcfg['significant_move_threshold']
event_times = game_odds.loc[event_mask, 'elapsed_seconds'].to_numpy()
T = float(game_odds['elapsed_seconds'].max())

print(f'Found {len(event_times)} significant odds moves in {T:.0f} seconds')

hawkes_model.fit(event_times, T)
print(f'Fitted: μ={hawkes_model.mu_:.4f}, α={hawkes_model.alpha_:.4f}, β={hawkes_model.beta_:.4f}')
print(f'Branching ratio: α/β = {hawkes_model.branching_ratio:.4f} (must be < 1)')

In [ ]:
# Plot intensity over the game
query_times = np.linspace(0, T, 300)
intensity_vals = hawkes_model.intensity_path(query_times, event_times)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(query_times, intensity_vals, color='#9467bd', lw=1.5, label='Hawkes λ(t)')
ax.scatter(event_times,
           hawkes_model.intensity_path(event_times, event_times),
           color='#9467bd', s=20, zorder=5, label='Significant moves')
ax.set_xlabel('Elapsed Seconds')
ax.set_ylabel('Intensity λ(t)')
ax.set_title(f'Hawkes Intensity — Game {sample_gid}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 2.4 Kalman Filter — Latent WP Extraction

In [ ]:
from src.models.kalman_filter import WinProbabilityFilter

kf_cfg = cfg['kalman']
kf = WinProbabilityFilter(
    Q=kf_cfg['Q'],
    R=kf_cfg['R'],
    x0=kf_cfg['x0'],
    P0=kf_cfg['P0'],
    scoring_Q_multiplier=kf_cfg['scoring_Q_multiplier'],
)

game_pbp = pbp_feat[pbp_feat['game_id'] == sample_gid].copy()
filtered_df = kf.filter_game(game_pbp, game_odds)
print(filtered_df.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax = axes[0]
ax.plot(filtered_df['elapsed_seconds'], filtered_df['nflfastr_wp'],
        label='nflfastR WP (true)', color='#1f77b4', lw=1.5)
ax.plot(filtered_df['elapsed_seconds'], filtered_df['filtered_wp'],
        label='Kalman Filtered WP', color='#ff7f0e', lw=2, zorder=3)
ax.plot(filtered_df['elapsed_seconds'], filtered_df['book_implied_prob'],
        label='Book Implied Prob', color='#2ca02c', lw=1.0, ls='--', alpha=0.7)
ax.set_ylabel('Win Probability')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.set_title(f'Kalman Filter Output — Game {sample_gid}')

ax = axes[1]
ax.fill_between(
    filtered_df['elapsed_seconds'],
    filtered_df['filtered_wp'] - 2 * np.sqrt(filtered_df['filtered_variance'].clip(0)),
    filtered_df['filtered_wp'] + 2 * np.sqrt(filtered_df['filtered_variance'].clip(0)),
    alpha=0.2, color='#ff7f0e', label='±2σ posterior band'
)
ax.plot(filtered_df['elapsed_seconds'], filtered_df['filtered_wp'],
        color='#ff7f0e', lw=1.5)
ax.set_xlabel('Elapsed Seconds')
ax.set_ylabel('Filtered WP')
ax.set_title('Posterior Win Probability with Uncertainty Band')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 2.5 Monte Carlo Game Simulation

In [ ]:
from src.simulation.monte_carlo import GameSimulator

mc_cfg = cfg['monte_carlo']
simulator = GameSimulator(scoring_model=scoring_model, n_sims=5000)

# Simulate from a mid-game state (half time, tied 10-10)
state = {
    'home_score': 10,
    'away_score': 10,
    'possession': 'home',
    'yardline_100': 60,
    'down': 1,
    'ydstogo': 10,
    'elapsed_seconds': 1800.0,
}
remaining = 1800.0  # second half

result = simulator.simulate_from_state(state, remaining, seed=123)
print(f"Home wins: {result['home_wins']:.3f}")
print(f"Away wins: {result['away_wins']:.3f}")
print(f"Ties:      {result['ties']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(result['home_score_dist'], bins=30, color='#1f77b4', edgecolor='white',
        alpha=0.7, label='Home')
ax.hist(result['away_score_dist'], bins=30, color='#d62728', edgecolor='white',
        alpha=0.5, label='Away')
ax.axvline(result['home_score_dist'].mean(), color='#1f77b4', ls='--', lw=2)
ax.axvline(result['away_score_dist'].mean(), color='#d62728', ls='--', lw=2)
ax.set_xlabel('Final Score')
ax.set_ylabel('Simulation Count')
ax.set_title('Score Distribution (Halftime, 10–10)')
ax.legend()

ax = axes[1]
score_diff = result['home_score_dist'] - result['away_score_dist']
ax.hist(score_diff, bins=40, color='#9467bd', edgecolor='white', alpha=0.7)
ax.axvline(0, color='black', lw=1)
ax.axvline(score_diff.mean(), color='#9467bd', ls='--', lw=2,
           label=f'Mean: {score_diff.mean():.1f}')
ax.set_xlabel('Home Score − Away Score')
ax.set_ylabel('Count')
ax.set_title('Score Differential Distribution')
ax.legend()

plt.tight_layout()
plt.show()

## 2.6 Generate Signals for the Sample Game

In [ ]:
from src.strategy.signal_generation import generate_signals

# Build MC fair odds at each play
plays = game_pbp.sort_values('elapsed_seconds')
sample_plays = plays.iloc[::5]  # every 5th play for speed

mc_records = []
for _, row in sample_plays.iterrows():
    elapsed = float(row['elapsed_seconds'])
    remaining = max(3600.0 - elapsed, 60.0)
    state = {
        'home_score': int(row.get('posteam_score', 0) or 0),
        'away_score': int(row.get('defteam_score', 0) or 0),
        'possession': 'home',
        'yardline_100': int(row.get('yardline_100', 75) or 75),
        'down': int(row.get('down', 1) or 1),
        'ydstogo': int(row.get('ydstogo', 10) or 10),
        'elapsed_seconds': elapsed,
    }
    mc_res = simulator.fair_odds(state, remaining, seed=42)
    mc_records.append({'elapsed_seconds': elapsed, **mc_res})

mc_df = pd.DataFrame(mc_records)

signals_df = generate_signals(
    filtered_df, mc_df,
    edge_threshold=cfg['signals']['edge_threshold']
)

print(signals_df['signal'].value_counts())
signals_df[signals_df['signal'] != 'no_bet'].head()

In [ ]:
from src.visualization.plots import plot_single_game

scoring_times = game_pbp[game_pbp['touchdown'].fillna(0) > 0]['elapsed_seconds'].to_numpy()

fig = plot_single_game(
    game_id=sample_gid,
    filtered_df=filtered_df,
    odds_df=game_odds,
    signals_df=signals_df,
    hawkes_times=event_times,
    hawkes_intensity=intensity_vals,
    scoring_times=scoring_times,
)
plt.show()

---
**Next:** Open `03_backtest_results.ipynb` for the full walk-forward backtest and performance analysis.